# Rocket Landing ML Model - ULTIMATE ROBUST VERSION

This notebook trains a highly robust neural network to predict optimal ignition altitudes.
It has been enhanced to handle:
1. **Extreme Initial Conditions**: High tumbling rates and large attitude errors (up to 45°).
2. **Sensor Noise**: Training features include realistic Gaussian noise to prevent overfitting to perfect state data.
3. **Massive Scale Range**: From Micro (0.2kg) to Super Heavy (150kg).
4. **Severe Weather**: Wind speeds up to 40 m/s with turbulence.
5. **Physical Uncertainties**: Randomized aerodynamic damping and actuator lag.

These improvements ensure the model can "fly on any kind of rocket in any reasonable conditions".

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import random
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, trapezoid
from scipy.interpolate import interp1d
from tqdm.notebook import tqdm
from datetime import datetime
import pickle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Set seeds (but allow some stochasticity for robustness)
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

In [ ]:
class PhysicsEngine:
    def __init__(self, config):
        self.config = config
        self.g = config.get('gravity', 9.81)
        self.rho_0 = config.get('air_density', 1.225)
        self.Cd = config.get('drag_coefficient', 0.5)
        self.A_ref = config.get('reference_area', 0.07068)
        self.wind_speed = config.get('wind_speed', 0.0)
        self.wind_direction = np.radians(config.get('wind_direction', 0.0))
        self.wind_model = config.get('wind_model', 'constant')
        self.turbulence = config.get('turbulence', 0.0)
        
    def get_air_density(self, altitude):
        H = 8500
        return self.rho_0 * np.exp(-max(0, altitude) / H)
    
    def get_wind_velocity(self, position, time):
        altitude = position[2]
        if self.wind_model == 'constant':
            wx = self.wind_speed * np.cos(self.wind_direction)
            wy = self.wind_speed * np.sin(self.wind_direction)
        elif self.wind_model == 'altitude_varying':
            h_ref = 10.0; alpha = 0.143
            factor = (max(altitude, 1.0) / h_ref) ** alpha
            wx = self.wind_speed * factor * np.cos(self.wind_direction)
            wy = self.wind_speed * factor * np.sin(self.wind_direction)
        elif self.wind_model == 'gusts':
            gust = self.wind_speed * 0.5 * np.sin(0.6 * time) + self.turbulence * np.random.normal()
            wx = (self.wind_speed + gust) * np.cos(self.wind_direction)
            wy = (self.wind_speed + gust) * np.sin(self.wind_direction)
        else:
            wx=0; wy=0
            
        return np.array([wx, wy, 0.0])

    def get_drag_force(self, velocity, position, time):
        rho = self.get_air_density(position[2])
        wind = self.get_wind_velocity(position, time)
        v_rel = velocity - wind
        v_mag = np.linalg.norm(v_rel)
        if v_mag < 0.01: return np.zeros(3)
        return -0.5 * rho * v_mag**2 * self.Cd * self.A_ref * (v_rel / v_mag)

    # Quaternion utilities
    def quaternion_to_rotation_matrix(self, q):
        w, x, y, z = q
        return np.array([
            [1 - 2*(y**2 + z**2), 2*(x*y - w*z), 2*(x*z + w*y)],
            [2*(x*y + w*z), 1 - 2*(x**2 + z**2), 2*(y*z - w*x)],
            [2*(x*z - w*y), 2*(y*z + w*x), 1 - 2*(x**2 + y**2)]
        ])

    def quaternion_to_euler(self, q):
        w, x, y, z = q
        roll = np.arctan2(2*(w*x + y*z), 1 - 2*(x**2 + y**2))
        pitch = np.arcsin(np.clip(2*(w*y - z*x), -1, 1))
        yaw = np.arctan2(2*(w*z + x*y), 1 - 2*(y**2 + z**2))
        return np.array([roll, pitch, yaw])

    def normalize_quaternion(self, q):
        norm = np.linalg.norm(q)
        return q / norm if norm > 1e-10 else np.array([1,0,0,0])

    def quaternion_multiply(self, q1, q2):
        w1, x1, y1, z1 = q1; w2, x2, y2, z2 = q2
        return np.array([w1*w2 - x1*x2 - y1*y2 - z1*z2, w1*x2 + x1*w2 + y1*z2 - z1*y2, 
                        w1*y2 - x1*z2 + y1*w2 + z1*x2, w1*z2 + x1*y2 - y1*x2 + z1*w2])
    
    def euler_to_quaternion(self, r, p, y):
        cr = np.cos(r*0.5); sr = np.sin(r*0.5)
        cp = np.cos(p*0.5); sp = np.sin(p*0.5)
        cy = np.cos(y*0.5); sy = np.sin(y*0.5)
        return np.array([
            cr*cp*cy + sr*sp*sy,
            sr*cp*cy - cr*sp*sy,
            cr*sp*cy + sr*cp*sy,
            cr*cp*sy - sr*sp*cy
        ])

In [ ]:
class SuicideBurnSimulation:
    def __init__(self, r_cfg, e_cfg, s_cfg):
        self.r_cfg = r_cfg; self.e_cfg = e_cfg; self.s_cfg = s_cfg
        self.physics = PhysicsEngine(e_cfg); self.motor = SolidMotor(r_cfg)
        self.dry_mass = r_cfg['dry_mass']
        self.prop_mass = r_cfg['propellant_mass']
        self.A_ref = e_cfg['reference_area']
        self.length = r_cfg.get('length', 2.0)
        self.diameter = r_cfg.get('diameter', 0.15)
        self.damping_factor = e_cfg.get('damping_factor', 0.5)
        self.d_mult = 1.0; self.t_mult = 1.0
        
        # PID Gains
        self.kp = r_cfg.get('tvc_kp_pitch', 1.2)   # Tuned defaults for 6-DOF
        self.ki = r_cfg.get('tvc_ki_pitch', 0.1)
        self.kd = r_cfg.get('tvc_kd_pitch', 0.4)
        
    def calculate_dynamic_cg(self, current_mass):
        fuel = max(0, current_mass - self.dry_mass)
        frac = fuel / self.prop_mass if self.prop_mass > 0 else 0
        cg_dry = -self.length * 0.4
        cg_wet = -self.length * 0.5
        return np.array([0, 0, cg_dry + (cg_wet - cg_dry) * frac])
    
    def calculate_dynamic_inertia(self, current_mass, cg):
        r = self.diameter / 2; L = self.length
        I_xx = current_mass * (3*r**2 + L**2) / 12
        I_zz = current_mass * r**2 / 2
        return np.diag([I_xx, I_xx, I_zz])
    
    def calculate_ignition_altitude(self, vz, z):
        m0 = self.dry_mass + self.prop_mass
        avg_thrust = self.motor.total_impulse / max(0.1, self.motor.burn_time)
        a_net = (avg_thrust / m0) - self.physics.g
        if a_net <= 0: return z 
        return max(5.0, (vz**2) / (2 * a_net))

    def state_derivative(self, t, y, is_powered):
        # y: [pos(3), vel(3), quat(4), omg(3), m(1), p_int(1), y_int(1), p_tvc(1), y_tvc(1)]
        pos, vel, q = y[0:3], y[3:6], y[6:10]
        omg, m = y[10:13], max(0.01, y[13])
        p_int, y_int = y[14], y[15]
        cur_p_tvc, cur_y_tvc = y[16], y[17]
        
        R = self.physics.quaternion_to_rotation_matrix(q)
        
        # Forces
        F_g = np.array([0, 0, -m*9.81])
        F_d = self.physics.get_drag_force(vel, pos, t) * self.d_mult
        
        # Thrust Vector using state-based TVC angles
        thrust_mag = self.motor.get_thrust(t)
        # Use current TVC state from y
        thrust_body = thrust_mag * np.array([np.sin(cur_p_tvc), -np.sin(cur_y_tvc), np.cos(cur_p_tvc)*np.cos(cur_y_tvc)])
        F_t = (R @ thrust_body) * self.t_mult
        
        acc = (F_g + F_d + F_t) / m
        
        # Moments
        cg = self.calculate_dynamic_cg(m)
        M_thrust = np.cross(cg, thrust_body) * self.t_mult
        M_damp = -self.damping_factor * omg * (1 + 0.05*np.linalg.norm(vel))
        M_total = M_thrust + M_damp
        
        I_body = self.calculate_dynamic_inertia(m, cg)
        try: omg_dot = np.linalg.solve(I_body, M_total - np.cross(omg, I_body @ omg))
        except: omg_dot = np.zeros(3)
        
        dq = 0.5 * self.physics.quaternion_multiply(q, [0, *omg])
        dm = -self.motor.get_mass_flow_rate(t)
        
        # Control Logic (Stateless implementation inside derivative)
        dp_int = 0; dy_int = 0; dp_tvc = 0; dy_tvc = 0
        if is_powered:
            euler = self.physics.quaternion_to_euler(q)
            roll, pitch, yaw = euler
            
            # PID outputs
            # Using body rates (omg) for D-term is much more stable
            p_err = -pitch; y_err = -yaw
            p_cmd = self.kp*p_err + self.ki*p_int - self.kd*omg[1]
            y_cmd = self.kp*y_err + self.ki*y_int - self.kd*omg[0] # Note axis mapping
            
            p_cmd = np.clip(p_cmd, -self.motor.tvc_max, self.motor.tvc_max)
            y_cmd = np.clip(y_cmd, -self.motor.tvc_max, self.motor.tvc_max)
            
            # Integrator derivatives
            dp_int = p_err if abs(p_int) < 0.5 else 0
            dy_int = y_err if abs(y_int) < 0.5 else 0
            
            # TVC Actuator lag: d(tvc)/dt = (cmd - cur) / tau
            tau = max(0.01, self.motor.tvc_response_time)
            dp_tvc = (p_cmd - cur_p_tvc) / tau
            dy_tvc = (y_cmd - cur_y_tvc) / tau
        
        return np.concatenate([vel, acc, dq, omg_dot, [dm, dp_int, dy_int, dp_tvc, dy_tvc]])

    def run_simulation(self, init_state, ign_alt=None, fixed_params=None):
        # init_state should have 14 elements (from generation call)
        # We expand it to 18 internally
        self.motor = SolidMotor(self.r_cfg)
        is_oracle_trial = fixed_params is not None
        
        if fixed_params: 
            self.d_mult = fixed_params.get('drag_multiplier', 1.0)
            self.t_mult = fixed_params.get('thrust_multiplier', 1.0)
            if 'dry_mass' in fixed_params: self.dry_mass = fixed_params['dry_mass']
            self.faults = None
        else: 
            self.d_mult = 1.0; self.t_mult = 1.0
            self.faults = FaultInjector(self.s_cfg.get('faults', {}))
            
        # Initialize 18-element state
        cur_y = np.zeros(18)
        cur_y[:14] = init_state
        
        est = StateEstimator(cur_y[13], self.physics.Cd)
        t_offset = 0; landed = False; is_pow = False
        hist = {'t':[], 'z':[], 'vz':[], 'vx':[], 'vy':[], 'roll':[], 'pitch':[], 'yaw':[], 
                'omega_x':[], 'omega_y':[], 'omega_z':[], 'mass':[], 'inf_m':[], 'inf_cd':[], 'y_full':[]}
        
        if ign_alt is None: 
            ign_alt = self.calculate_ignition_altitude(cur_y[5], cur_y[2])
            
        while t_offset < 90 and not landed: 
            t_target = 90; pend = [f for f in self.faults.events if f['time'] > t_offset] if self.faults else []
            if pend: t_target = pend[0]['time']
            
            if t_target <= t_offset: t_target = t_offset + 5.0 # Safety
            
            def gnd(t, y): return y[2]
            gnd.terminal = True; gnd.direction = -1
            def ign(t, y): return y[2] - ign_alt if not is_pow else 1.0
            ign.terminal = True; ign.direction = -1
            
            # solve_ivp is now calling a COMPLETELY STATELESS derivative
            sol = solve_ivp(self.state_derivative, [t_offset, t_target], cur_y, 
                          args=(is_pow,), events=[gnd, ign], rtol=1e-3, atol=1e-3, max_step=0.2) 
            
            if not is_oracle_trial:
                for i in range(len(sol.t)):
                    y_ptr = sol.y[:, i]; t_ptr = sol.t[i]
                    R_mat = self.physics.quaternion_to_rotation_matrix(y_ptr[6:10])
                    # Est compute
                    t_mag = self.motor.get_thrust(t_ptr)
                    t_body = t_mag * np.array([np.sin(y_ptr[16]), -np.sin(y_ptr[17]), np.cos(y_ptr[16])*np.cos(y_ptr[17])])
                    thr_vec = R_mat @ t_body * self.t_mult
                    thr_z_local = (R_mat.T @ thr_vec)[2]
                    
                    az_meas = self.state_derivative(t_ptr, y_ptr, is_pow)[5] + 9.81 + np.random.normal(0, 0.3) 
                    vz_meas = y_ptr[5] + np.random.normal(0, 0.1)
                    est_s = est.update(az_meas, vz_meas, self.physics.get_air_density(y_ptr[2]), thr_z_local)
                    est.predict(self.motor.get_mass_flow_rate(t_ptr))
                    
                    hist['t'].append(t_ptr)
                    hist['z'].append(y_ptr[2]); hist['vz'].append(y_ptr[5])
                    hist['vx'].append(y_ptr[3]); hist['vy'].append(y_ptr[4])
                    eu = self.physics.quaternion_to_euler(y_ptr[6:10])
                    hist['roll'].append(eu[0]); hist['pitch'].append(eu[1]); hist['yaw'].append(eu[2])
                    hist['omega_x'].append(y_ptr[10]); hist['omega_y'].append(y_ptr[11]); hist['omega_z'].append(y_ptr[12])
                    hist['mass'].append(y_ptr[13])
                    hist['inf_m'].append(est_s[0]); hist['inf_cd'].append(est_s[1])
                    hist['y_full'].append(y_ptr[:14]) # Store 14-element for ML compatibility
                
            cur_y = sol.y[:, -1]; t_offset = sol.t[-1]
            cur_y[6:10] = self.physics.normalize_quaternion(cur_y[6:10])
            
            if len(sol.t_events[0]) > 0: landed = True
            if not is_pow and len(sol.t_events)>1 and len(sol.t_events[1])>0:
                is_pow = True; self.motor.ignite(t_offset)
                
            if self.faults and pend and abs(t_offset - t_target) < 1e-3:
                f = pend[0]
                if f['type'] == 'mass_drop': cur_y[13] -= f['val']
                elif f['type'] == 'drag_change': self.d_mult = f['val']
                elif f['type'] == 'thrust_anomaly': self.t_mult = 1.0/f['val']
                
        pitch_final = self.physics.quaternion_to_euler(cur_y[6:10])[1]
        success = (abs(cur_y[5]) < 5.0) and (abs(pitch_final) < 0.6)
        return success, cur_y[:14], hist

    def optimize_oracle(self, state, estimated_params):
        guess = self.calculate_ignition_altitude(state[5], state[2])
        best_alt = guess; best_cost = 9999
        
        for rng, stp in [(80, 20), (25, 5), (6, 1.5)]:
            space = np.arange(max(5, best_alt-rng), best_alt+rng, stp)
            for alt in space:
                _, fin, _ = self.run_simulation(state.copy(), alt, fixed_params=estimated_params)
                cost = abs(fin[5]) + abs(fin[2])*0.2
                if cost < best_cost: best_cost = cost; best_alt = alt
        return best_alt


In [ ]:
def generate_robust_data(num_flights=500):
    dataset = []
    classes = [
        {'name': 'Micro', 'dry': (0.1, 0.3), 'prop': (0.05, 0.15), 'd': 0.03, 'l': (0.3, 0.6), 'p': (30, 80), 'b': (0.5, 1.2)},
        {'name': 'Small', 'dry': (0.3, 0.9), 'prop': (0.1, 0.3), 'd': 0.05, 'l': (0.6, 1.2), 'p': (80, 200), 'b': (1.0, 2.0)},
        {'name': 'Medium', 'dry': (1.2, 4.0), 'prop': (0.3, 1.0), 'd': 0.08, 'l': (1.5, 2.5), 'p': (350, 900), 'b': (2.5, 5.0)},
        {'name': 'Large', 'dry': (7.0, 18.0), 'prop': (1.5, 4.0), 'd': 0.15, 'l': (3.0, 5.0), 'p': (1800, 4000), 'b': (4.0, 9.0)},
        {'name': 'Heavy', 'dry': (20.0, 60.0), 'prop': (4.0, 12.0), 'd': 0.25, 'l': (5.0, 8.0), 'p': (4000, 10000), 'b': (6.0, 15.0)},
        {'name': 'SuperHeavy', 'dry': (60.0, 150.0), 'prop': (15.0, 50.0), 'd': 0.5, 'l': (10.0, 20.0), 'p': (15000, 40000), 'b': (10.0, 25.0)}
    ]
    
    print(f"Generating data (18-element state vector, stateless ODE)...")
    for f_idx in tqdm(range(num_flights)):
        cls = random.choice(classes)
        dry = random.uniform(*cls['dry']); prop = random.uniform(*cls['prop'])
        dia = cls['d'] * random.uniform(0.9, 1.1)
        length = random.uniform(*cls['l'])
        peak = random.uniform(*cls['p']); burn = random.uniform(*cls['b'])
        
        wind_v = random.uniform(0, 35)
        wind_m = random.choice(['constant', 'altitude_varying', 'gusts'])
        
        tc = [[0,0], [0.1, peak], [burn, peak], [burn+0.1, 0]]
        
        r_cfg = {
            'dry_mass': dry, 'propellant_mass': prop, 'thrust_curve': tc,
            'length': length, 'diameter': dia, 'tvc_max_angle': np.radians(random.uniform(3, 8)),
            'tvc_response_time': random.uniform(0.08, 0.25)
        }
        e_cfg = {
            'gravity': 9.81, 'air_density': random.uniform(1.1, 1.3), 'drag_coefficient': random.uniform(0.4, 0.7),
            'reference_area': np.pi*(dia/2)**2, 'wind_speed': wind_v, 'wind_direction': random.uniform(0, 360),
            'wind_model': wind_m, 'damping_factor': random.uniform(0.2, 0.8)
        }
        s_cfg = {'faults': {'mass_drop_prob': 0.15, 'drag_change_prob': 0.15, 'thrust_anomaly_prob': 0.15}}
        
        sim = SuicideBurnSimulation(r_cfg, e_cfg, s_cfg)
        h0 = random.uniform(300, 1500)
        v0 = random.uniform(-10, -100)
        
        ir, ip, iy = [random.uniform(-0.6, 0.6) for _ in range(3)]
        q0 = sim.physics.euler_to_quaternion(ir, ip, iy)
        omg0 = [random.uniform(-1.5, 1.5) for _ in range(3)]
        
        init_st = np.array([0, 0, h0, 0, 0, v0, *q0, *omg0, dry+prop])
        success, final, hist = sim.run_simulation(init_st)
        
        ascent_twr = peak / ((dry+prop)*9.81)
        
        # Use fixed-size sampling for speed and balance
        n_steps = len(hist['t'])
        if n_steps > 30:
            indices = np.linspace(10, n_steps-5, 15, dtype=int)
        else:
            indices = range(n_steps)

        for i in indices:
            if hist['z'][i] < 4 or hist['vz'][i] > -5: continue
            
            # Oracle search for perfect burn altitude
            est_p = {'drag_multiplier':1.0, 'thrust_multiplier':1.0, 'dry_mass': hist['inf_m'][i]-prop}
            opt_alt = sim.optimize_oracle(hist['y_full'][i], est_p)
            
            z_noisy = hist['z'][i] + np.random.normal(0, 1.5)
            vz_noisy = hist['vz'][i] + np.random.normal(0, 0.3)
            rho = sim.physics.get_air_density(z_noisy)
            
            dataset.append({
                'rocket_class': cls['name'],
                'ascent_twr': ascent_twr, 'rocket_mass': hist['inf_m'][i],
                'current_altitude': z_noisy, 'descent_velocity': vz_noisy,
                'dynamic_pressure': 0.5 * rho * vz_noisy**2,
                'pitch_angle': hist['pitch'][i] + np.random.normal(0, 0.02),
                'roll_angle_abs': abs(hist['roll'][i]),
                'omega_z': hist['omega_z'][i] + np.random.normal(0, 0.01),
                'inferred_drag_coeff': hist['inf_cd'][i], 'wind_speed': wind_v,
                'predicted_ignition_altitude': sim.calculate_ignition_altitude(vz_noisy, z_noisy),
                'TARGET_optimal_ignition_altitude': opt_alt
            })
            
    df_res = pd.DataFrame(dataset)
    df_res.to_csv('robust_training_data_v2.csv', index=False)
    return df_res


In [ ]:
def generate_robust_data(num_flights=500):
    dataset = []
    classes = [
        {'name': 'Micro', 'dry': (0.1, 0.3), 'prop': (0.05, 0.15), 'd': 0.03, 'l': (0.3, 0.6), 'p': (30, 80), 'b': (0.5, 1.2)},
        {'name': 'Small', 'dry': (0.3, 0.9), 'prop': (0.1, 0.3), 'd': 0.05, 'l': (0.6, 1.2), 'p': (80, 200), 'b': (1.0, 2.0)},
        {'name': 'Medium', 'dry': (1.2, 4.0), 'prop': (0.3, 1.0), 'd': 0.08, 'l': (1.5, 2.5), 'p': (350, 900), 'b': (2.5, 5.0)},
        {'name': 'Large', 'dry': (7.0, 18.0), 'prop': (1.5, 4.0), 'd': 0.15, 'l': (3.0, 5.0), 'p': (1800, 4000), 'b': (4.0, 9.0)},
        {'name': 'Heavy', 'dry': (20.0, 60.0), 'prop': (4.0, 12.0), 'd': 0.25, 'l': (5.0, 8.0), 'p': (4000, 10000), 'b': (6.0, 15.0)},
        {'name': 'SuperHeavy', 'dry': (60.0, 150.0), 'prop': (15.0, 50.0), 'd': 0.5, 'l': (10.0, 20.0), 'p': (15000, 40000), 'b': (10.0, 25.0)}
    ]
    
    print(f"Generating data with robust randomization...")
    for _ in tqdm(range(num_flights)):
        cls = random.choice(classes)
        dry = random.uniform(*cls['dry']); prop = random.uniform(*cls['prop'])
        dia = cls['d'] * random.uniform(0.9, 1.1)
        length = random.uniform(*cls['l'])
        peak = random.uniform(*cls['p']); burn = random.uniform(*cls['b'])
        
        # Wind up to 40 m/s (Hurricane I)
        wind_v = random.uniform(0, 40)
        wind_m = random.choice(['constant', 'altitude_varying', 'gusts'])
        
        tc = [[0,0], [0.1, peak], [burn, peak], [burn+0.1, 0]]
        
        r_cfg = {
            'dry_mass': dry, 'propellant_mass': prop, 'thrust_curve': tc,
            'length': length, 'diameter': dia, 'tvc_max_angle': random.uniform(3, 10),
            'tvc_response_time': random.uniform(0.05, 0.3)
        }
        e_cfg = {
            'gravity': 9.81, 'air_density': random.uniform(1.0, 1.4), 'drag_coefficient': random.uniform(0.3, 0.9),
            'reference_area': np.pi*(dia/2)**2, 'wind_speed': wind_v, 'wind_direction': random.uniform(0, 360),
            'wind_model': wind_m, 'damping_factor': random.uniform(0.1, 1.0)
        }
        s_cfg = {'faults': {'mass_drop_prob': 0.2, 'drag_change_prob': 0.2, 'thrust_anomaly_prob': 0.2}} # Higher fault prob
        
        sim = SuicideBurnSimulation(r_cfg, e_cfg, s_cfg)
        h0 = random.uniform(300, 1500)
        v0 = random.uniform(-10, -120)
        
        # EXTREME ATTITUDE (up to 45 deg error, tumbling 2 rad/s)
        ir, ip, iy = [random.uniform(-0.8, 0.8) for _ in range(3)]
        q0 = sim.physics.euler_to_quaternion(ir, ip, iy)
        omg = [random.uniform(-2, 2) for _ in range(3)]
        
        init_st = np.array([0, 0, h0, 0, 0, v0, *q0, *omg, dry+prop])
        success, final, hist = sim.run_simulation(init_st)
        
        ascent_twr = peak / ((dry+prop)*9.81)
        
        for i in range(5, len(hist['t']), 8): # Subsample
            if hist['z'][i] < 3: continue
            if hist['vz'][i] > -5: continue
            
            # Reconstruction of estimated params
            est_p = {'drag_multiplier':1.0, 'thrust_multiplier':1.0, 'dry_mass': hist['inf_m'][i]-prop}
            
            # Oracle gets the physics-perfect simulation to find the "Physically Possible" solution
            opt_alt = sim.optimize_oracle(hist['y_full'][i], est_p)
            
            # ADD SENSOR NOISE TO FEATURES
            # The model sees NOISY data, but targets PERFECT execution
            z_noisy = hist['z'][i] + np.random.normal(0, 2.0) # Altimeter noise
            vz_noisy = hist['vz'][i] + np.random.normal(0, 0.5) # Doppler noise
            rho = sim.physics.get_air_density(z_noisy)
            
            dataset.append({
                'rocket_class': cls['name'],
                'ascent_twr': ascent_twr,
                'rocket_mass': hist['inf_m'][i],
                'current_altitude': z_noisy,
                'descent_velocity': vz_noisy,
                'dynamic_pressure': 0.5 * rho * vz_noisy**2, # NEW FEATURE
                'pitch_angle': hist['pitch'][i] + np.random.normal(0, 0.05),
                'roll_angle_abs': abs(hist['roll'][i]), # Absolute values often help
                'omega_z': hist['omega_z'][i] + np.random.normal(0, 0.01),
                'inferred_drag_coeff': hist['inf_cd'][i],
                'wind_speed': wind_v,
                'predicted_ignition_altitude': sim.calculate_ignition_altitude(vz_noisy, z_noisy),
                'TARGET_optimal_ignition_altitude': opt_alt
            })
            
    df = pd.DataFrame(dataset)
    df.to_csv('robust_training_data.csv', index=False)
    return df

In [ ]:
df = generate_robust_data(num_flights=600)
print(df.describe())

features = ['ascent_twr', 'rocket_mass', 'current_altitude', 'descent_velocity', 
            'dynamic_pressure', 'pitch_angle', 'omega_z', 'inferred_drag_coeff', 
            'wind_speed', 'predicted_ignition_altitude']

X = df[features].values
y = df['TARGET_optimal_ignition_altitude'].values

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_s, y, test_size=0.2, random_state=42)

model = keras.Sequential([
    layers.Dense(256, input_shape=[len(features)]),
    layers.BatchNormalization(),
    layers.LeakyReLU(alpha=0.1),
    layers.Dropout(0.2),
    
    layers.Dense(128),
    layers.BatchNormalization(),
    layers.LeakyReLU(alpha=0.1),
    layers.Dropout(0.2),
    
    layers.Dense(64),
    layers.LeakyReLU(alpha=0.1),
    
    layers.Dense(1)
])

model.compile(loss='huber', optimizer=keras.optimizers.Adam(learning_rate=0.001), metrics=['mae'])

hist = model.fit(X_train, y_train, validation_data=(X_test, y_test), 
                 epochs=300, batch_size=128, verbose=1,
                 callbacks=[keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True)])

model.save('robust_ignition_model.keras')
print("Robust Model Saved.")

Generating data with robust randomization...


  0%|          | 0/600 [00:00<?, ?it/s]

KeyboardInterrupt: 